# How has collaboration in scientific research changed across arXiv fields over time? 

## 3. Are researchers increasingly collaborating with authors whose research interests are different from their own?
Are researchers increasingly collaborating with authors whose previous research topics are different from their own?




## Set Up File Paths

The cleaned CSV is in `data/processed`. If it is missing, incomplete, or uses older sampling settings, the notebook rebuilds it from the raw arXiv metadata using the cleaning script in `scripts`.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    for folder in [start, *start.parents]:
        if (folder / "data").exists() and (folder / "scripts").exists():
            return folder
    raise FileNotFoundError("Could not find the project folder with data/ and scripts/.")


PROJECT_ROOT = find_project_root()
RAW_FILE = PROJECT_ROOT / "data" / "raw" / "arxiv-metadata-oai-snapshot.zip"
CLEAN_FILE = PROJECT_ROOT / "data" / "processed" / "arxiv_metadata_clean.csv"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

print(f"Project folder: {PROJECT_ROOT}")
print(f"Clean data file: {CLEAN_FILE}")

## Make Sure the Clean Data Exists

This step keeps up to 1,000 papers per year and primary category with complete essential fields, including authors. It reuses a completed sample and the existing raw download.


In [ ]:
import importlib
import clean_arxiv

clean_arxiv = importlib.reload(clean_arxiv)

if not clean_arxiv.clean_file_is_current(CLEAN_FILE):
    clean_arxiv.run_full(raw_file=RAW_FILE, clean_file=CLEAN_FILE)
else:
    print("Complete year/category sample already exists")

---
## Part 3 — Collaboration between authors from different research fields

**Run instructions:** The five cells above are copied from your original notebook without changing their code. The original fifth cell calls `clean_arxiv.run_full`, which may fail with a Kaggle 404. If you already have `data/processed/arxiv_metadata_clean.csv`, **skip that fifth cell** and run the analysis below. Otherwise, download the arXiv metadata separately, run your existing cleaning script, and then return here.

**Definition:** We use authors' *earlier publications in this dataset* as a proxy for research interests. A cross-field author pair has known prior work but **no broad arXiv field in common**. This is not a direct measure of personal interests; missing publication histories and author-name ambiguity affect the results.

In [ ]:
# Load the existing cleaned CSV without downloading anything.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from itertools import combinations
import ast
import re

if not CLEAN_FILE.is_file():
    raise FileNotFoundError(
        f"Clean CSV missing: {CLEAN_FILE}\n"
        "Download/clean the arXiv metadata using your project's script first. "
        "Do not run the original auto-download cell if Kaggle returns HTTP 404."
    )

df = pd.read_csv(CLEAN_FILE, low_memory=False)
required = {'submitted_year', 'authors', 'categories', 'primary_category'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Missing expected columns: {sorted(missing)}')
print('Loaded', f'{len(df):,}', 'papers from', CLEAN_FILE)
print('Years:', df.submitted_year.min(), 'to', df.submitted_year.max())

## Prepare authors and fields

Use broad arXiv categories such as `cs`, `math`, and `physics`. The raw author field may contain comma-separated names or a Python-style list; we handle both. Names are not reliable unique author IDs.

In [ ]:
def parse_authors(value):
    if isinstance(value, (list, tuple)):
        names = value
    elif pd.isna(value):
        return []
    else:
        s = str(value).strip()
        if s.startswith('[') and s.endswith(']'):
            try:
                parsed = ast.literal_eval(s)
                names = parsed if isinstance(parsed, (list, tuple)) else [s]
            except (SyntaxError, ValueError):
                names = s.split(',')
        else:
            names = s.split(',')
    return list(dict.fromkeys(str(a).strip().casefold() for a in names if str(a).strip()))


def broad_field(cat):
    cat = str(cat).strip()
    if not cat or cat.lower() == 'nan':
        return None
    # Legacy physics categories have no dot and are kept as their own groups.
    return cat.split('.')[0]


def parse_fields(value):
    if pd.isna(value):
        return set()
    return {field for cat in str(value).replace(',', ' ').split()
            if (field := broad_field(cat)) is not None}

work = df.copy()
work['year'] = pd.to_numeric(work['submitted_year'], errors='coerce')
work = work.dropna(subset=['year']).copy()
work['year'] = work['year'].astype(int)
work['author_list'] = work['authors'].map(parse_authors)
work['fields'] = work['categories'].map(parse_fields)
work['main_field'] = work['primary_category'].map(broad_field)
work = work[(work.author_list.map(len) > 0) & (work.fields.map(len) > 0)].copy()
if 'paper_id' in work:
    work = work.drop_duplicates(subset=['paper_id'])
print(f'Papers with authors and fields: {len(work):,}')
print(f'Multi-author papers: {(work.author_list.map(len) >= 2).sum():,}')

## Compare collaborators' previous research fields

For each year, compare authors' research profiles from **earlier years only**. A paper is comparable if at least one coauthor pair has earlier research histories. Count a paper as cross-field if any comparable pair has no overlapping previous broad field.

In [ ]:
profiles = defaultdict(set)
records = []

for year, papers in work.groupby('year', sort=True):
    # Evaluate the whole year against the profile from PREVIOUS years.
    for row in papers.itertuples(index=False):
        authors = row.author_list
        if len(authors) < 2:
            continue
        comparable = 0
        cross_field = 0
        for a, b in combinations(authors, 2):
            pa, pb = profiles.get(a, set()), profiles.get(b, set())
            if pa and pb:
                comparable += 1
                if pa.isdisjoint(pb):
                    cross_field += 1
        records.append({
            'year': year,
            'main_field': row.main_field,
            'comparable_pairs': comparable,
            'cross_field_pairs': cross_field,
            'comparable_paper': int(comparable > 0),
            'cross_field_paper': int(cross_field > 0),
        })
    # Update only AFTER measuring all collaborations from this year.
    for row in papers.itertuples(index=False):
        for author in row.author_list:
            profiles[author].update(row.fields)

results = pd.DataFrame.from_records(records)
if results.empty:
    raise ValueError('No multi-author papers found in the cleaned dataset.')
print('Multi-author papers examined:', f'{len(results):,}')
print('Papers with comparable author histories:', f'{results.comparable_paper.sum():,}')

## Yearly trends and visualizations

In [ ]:
yearly = results.groupby('year', as_index=False).agg(
    multi_author_papers=('comparable_paper', 'size'),
    comparable_papers=('comparable_paper', 'sum'),
    cross_field_papers=('cross_field_paper', 'sum'),
    comparable_pairs=('comparable_pairs', 'sum'),
    cross_field_pairs=('cross_field_pairs', 'sum'),
)
yearly['cross_field_paper_pct'] = 100 * yearly['cross_field_papers'] / yearly['comparable_papers'].replace(0, np.nan)
yearly['cross_field_pair_pct'] = 100 * yearly['cross_field_pairs'] / yearly['comparable_pairs'].replace(0, np.nan)
display(yearly.tail(15))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(yearly.year, yearly.cross_field_paper_pct, marker='o', label='Papers with cross-field pair (%)')
ax.plot(yearly.year, yearly.cross_field_pair_pct, marker='s', label='Cross-field author pairs (%)')
ax.set(xlabel='Year', ylabel='Percent of comparable collaborations',
       title='Cross-field collaboration over time (prior publication histories)')
ax.legend()
ax.grid(alpha=.3)
plt.tight_layout()
plt.show()

## Compare arXiv fields

In [ ]:
by_field = results.groupby('main_field', dropna=False).agg(
    comparable_papers=('comparable_paper', 'sum'),
    cross_field_papers=('cross_field_paper', 'sum'),
    comparable_pairs=('comparable_pairs', 'sum'),
    cross_field_pairs=('cross_field_pairs', 'sum'),
).reset_index()
by_field['cross_field_paper_pct'] = 100 * by_field.cross_field_papers / by_field.comparable_papers.replace(0, np.nan)
by_field = by_field.sort_values('comparable_papers', ascending=False)
display(by_field.head(15))

plot_fields = by_field[by_field.comparable_papers >= 20].head(12)
if not plot_fields.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(plot_fields.main_field.astype(str), plot_fields.cross_field_paper_pct)
    ax.set(xlabel='Primary broad field', ylabel='Cross-field papers (%)',
           title='Cross-field collaboration by primary arXiv field')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## Interpretation and save results

The trend is descriptive, not causal. Early years may have few comparable author histories. Sampling, incomplete arXiv coverage, shared names, and broad field definitions can change the observed trend.

In [ ]:
valid = yearly.dropna(subset=['cross_field_paper_pct'])
valid = valid[valid.comparable_papers >= 20]
if len(valid) >= 2:
    first, last = valid.iloc[0], valid.iloc[-1]
    delta = last.cross_field_paper_pct - first.cross_field_paper_pct
    direction = 'increased' if delta > 0 else ('decreased' if delta < 0 else 'stayed the same')
    print(f'Between {int(first.year)} and {int(last.year)}, the share of comparable papers '
          f'with a cross-field coauthor pair {direction} from '
          f'{first.cross_field_paper_pct:.1f}% to {last.cross_field_paper_pct:.1f}% '
          f'({delta:+.1f} percentage points).')
else:
    print('Not enough years with 20+ comparable papers to describe a reliable trend.')

output_dir = PROJECT_ROOT / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
yearly.to_csv(output_dir / 'part3_yearly_results.csv', index=False)
by_field.to_csv(output_dir / 'part3_field_results.csv', index=False)
print('Saved CSV results to:', output_dir)